[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaytAlhikmah/hands-on-llms-for-swes/blob/main/sessions/2/notebook.ipynb)

# Session 2: Information Theory Fundamentals

**Read this alongside `lesson.md`** — this notebook contains exercises referenced in the lesson.

In session 1, you saw that language models predict the next token. But behind every prediction is a deeper question: how much information does language contain? How efficiently can we encode it? What happens when our assumptions about the data are wrong?

This session answers those questions using Shannon's information theory — first measuring unpredictability (entropy), then measuring coding inefficiency (cross-entropy), and finally quantifying wasted bits (KL divergence).

---

## Part 0: Understanding Entropy

_Corresponds to lesson sections 0-3_

Before we can measure coding efficiency, we need to understand **entropy** — Shannon's measure of unpredictability and the theoretical limit of compression.

### Setup

In [ ]:
# Install the course package
!pip install -q git+https://github.com/BaytAlhikmah/hands-on-llms-for-swes.git#subdirectory=pkg

print('✓ Setup complete!')

### Exercise 1: Binary Decision Tree (Uniform Case)

_Lesson section 1: The guessing game with 8 equally likely options_

Visualize how binary search works when all outcomes are equally likely. With 8 options, you always need exactly log₂(8) = 3 questions.

In [ ]:
import matplotlib.pyplot as plt
from alhikmah_llms import Session2

# Visualize the uniform case: 8 equally likely outcomes
Session2.draw_uniform_tree()
plt.show()

print("Entropy = log₂(8) = 3.0 bits")
print("Every path from root to leaf takes exactly 3 questions.")

### Exercise 2: Binary Decision Tree (Biased Case)

_Lesson section 1: The guessing game with biased probabilities_

When outcomes aren't equally likely, you can do better! Check the most probable outcome first.

In [ ]:
# Visualize the biased case: 50% on option 1, rest split among 2-8
Session2.draw_biased_tree()
plt.show()

print("Entropy ≈ 2.4035 bits")
print("50% of the time, you're done in 1 question!")
print("Average: 0.5 × 1 + 0.5 × 3.807 = 2.4035 questions")

### Exercise 3: Visualize Probability Distributions

_Lesson section 1: How entropy relates to the "spread" of probabilities_

The more "spiky" the distribution, the lower the entropy.

In [ ]:
# Compare the distributions
Session2.plot_distributions()
plt.show()

print("Key Observation:")
print("   Uniform (flat) = maximum entropy = maximum unpredictability")
print("   Biased (spiky) = lower entropy = more predictable")

### Exercise 4: Compute Entropy for Different Distributions

_Lesson section 2: Deriving the entropy formula from first principles_

Verify that the formula H = Σ p(x) log₂(1/p(x)) matches our intuition.

In [ ]:
import math

def entropy(probs: list[float]) -> float:
    """Average number of yes/no questions (bits) to guess the outcome."""
    return sum(p * math.log2(1/p) for p in probs if p > 0)

# Test the formula
uniform_8 = [1/8] * 8
biased = [0.5] + [0.5/7] * 7
certain = [1.0] + [0.0] * 7

print(f"Uniform (all equal):     {entropy(uniform_8):.4f} bits  (= log₂(8))")
print(f"Biased (50% on one):     {entropy(biased):.4f} bits")
print(f"Certain (100% on one):   {entropy(certain):.4f} bits")
print()
print("Lower entropy = more predictable = fewer questions needed")

### Exercise 5: Implement and Test the Entropy Function

_Lesson section 2: Understanding the two equivalent forms_

The formula can be written as -Σ p log₂(p) or Σ p log₂(1/p). Why are they equivalent?

In [ ]:
# Two equivalent forms
def entropy_form1(probs: list[float]) -> float:
    """Form 1: Σ p(x) × log₂(1/p(x))"""
    return sum(p * math.log2(1/p) for p in probs if p > 0)

def entropy_form2(probs: list[float]) -> float:
    """Form 2: -Σ p(x) × log₂(p(x))"""
    return -sum(p * math.log2(p) for p in probs if p > 0)

# Verify they're the same
test_probs = [0.6, 0.2, 0.1, 0.1]
h1 = entropy_form1(test_probs)
h2 = entropy_form2(test_probs)

print(f"Form 1: {h1:.6f} bits")
print(f"Form 2: {h2:.6f} bits")
print(f"Difference: {abs(h1 - h2):.10f}")
print()
print("Why equivalent? Because log₂(1/p) = log₂(1) - log₂(p) = 0 - log₂(p) = -log₂(p)")

### Exercise 6: Compression Visualization

_Lesson section 3B: Entropy as optimal compression size_

See how entropy-based encoding saves space by assigning short codes to frequent symbols.

In [ ]:
Session2.plot_compression_example()
plt.show()

print("Key Insight:")
print("   Entropy = theoretical compression limit")
print("   Frequent symbols get short codes, rare symbols get long codes")

### Exercise 7: Surprise Curve

_Lesson section 3C: Entropy as average surprise_

See how surprise = -log₂(p) relates to probability.

In [ ]:
Session2.plot_surprise_curve()
plt.show()

print("Key Insight:")
print("   High probability → low surprise (you expected it)")
print("   Low probability → high surprise (you didn't see that coming!)")
print("   Entropy = expected surprise across all possible outcomes")

### Exercise 8: Explore Entropy Across Distributions

_Lesson section 3: Interactive visualizations_

#### 8a. Binary Entropy Function

The most fundamental curve in information theory — entropy of a coin flip with probability p.

In [ ]:
Session2.plot_binary_entropy()
plt.show()

print("Key Observation:")
print("   Maximum entropy at p=0.5 (fair coin flip)")
print("   Zero entropy at p=0 or p=1 (certain outcome)")
print("   Symmetric curve — this is the foundation of all entropy calculations!")

#### 8b. Ternary Entropy (3-outcome distributions)

Shows entropy for all possible 3-outcome probability distributions.

In [ ]:
Session2.plot_ternary_entropy_3d_plotly().show()

print("Key Observation:")
print("   Click and drag to rotate the 3D surface!")
print("   Peak at center (uniform 1/3, 1/3, 1/3) = maximum entropy ≈ 1.585 bits")
print("   Valleys at corners (certain outcome) = zero entropy")

#### 8c. Mathematical Properties of Entropy

Now that you've seen entropy in action, let's formalize its key mathematical properties. These properties uniquely characterize the entropy function and explain why it's the "right" measure of uncertainty.

**Property 1: Non-negativity**
```
H(X) ≥ 0
```
Entropy is always non-negative. It equals zero only when one outcome has probability 1 (complete certainty).

**Property 2: Maximum Entropy**
```
H(X) ≤ log₂(n)
```
For n possible outcomes, entropy is maximized when all outcomes are equally likely (uniform distribution). Maximum = log₂(n) bits.

**Property 3: Continuity**
H is a continuous function of the probability distribution. Small changes in probabilities lead to small changes in entropy.

**Property 4: Symmetry**
H is invariant under permutation of outcomes. The order doesn't matter, only the probabilities.

**Property 5: Additivity (for independent events)**
```
H(X,Y) = H(X) + H(Y)  [if X and Y are independent]
```
The joint entropy of two independent random variables equals the sum of their individual entropies.

**Property 6: Chain Rule**
```
H(X,Y) = H(X) + H(Y|X)
```
Joint entropy equals the entropy of X plus the conditional entropy of Y given X.

**Property 7: Conditioning Reduces Entropy**
```
H(Y|X) ≤ H(Y)
```
Knowing X can only reduce (or leave unchanged) uncertainty about Y. Information never increases uncertainty. Equality holds when X and Y are independent.

These properties aren't just mathematical curiosities — they capture fundamental facts about information and uncertainty.

In [ ]:
# Demonstrate the properties with examples

# Property 1: Non-negativity
print("Property 1: Non-negativity")
print("-" * 50)
certain = [1.0, 0.0, 0.0]
uniform = [1/3, 1/3, 1/3]
biased = [0.7, 0.2, 0.1]

print(f"  Certain outcome:  H = {entropy(certain):.4f} bits  (minimum)")
print(f"  Uniform:          H = {entropy(uniform):.4f} bits")
print(f"  Biased:           H = {entropy(biased):.4f} bits")
print(f"  All ≥ 0 ✓\n")

# Property 2: Maximum Entropy
print("Property 2: Maximum Entropy")
print("-" * 50)
n = 8
uniform_8 = [1/8] * 8
max_entropy = math.log2(n)
actual_entropy = entropy(uniform_8)
print(f"  For n={n} outcomes:")
print(f"  Maximum possible:  log₂({n}) = {max_entropy:.4f} bits")
print(f"  Uniform achieves:  H = {actual_entropy:.4f} bits")
print(f"  Any other distribution will have H < {max_entropy:.4f}")

# Test with non-uniform
non_uniform = [0.5, 0.1, 0.1, 0.1, 0.1, 0.05, 0.03, 0.02]
print(f"  Non-uniform:       H = {entropy(non_uniform):.4f} bits < {max_entropy:.4f} ✓\n")

# Property 4: Symmetry
print("Property 4: Symmetry")
print("-" * 50)
dist1 = [0.5, 0.3, 0.2]
dist2 = [0.2, 0.5, 0.3]  # Same probabilities, different order
dist3 = [0.3, 0.2, 0.5]  # Another permutation
print(f"  [0.5, 0.3, 0.2] → H = {entropy(dist1):.6f} bits")
print(f"  [0.2, 0.5, 0.3] → H = {entropy(dist2):.6f} bits")
print(f"  [0.3, 0.2, 0.5] → H = {entropy(dist3):.6f} bits")
print(f"  Order doesn't matter, only the probabilities ✓\n")

# Property 5: Additivity (independent events)
print("Property 5: Additivity for Independent Events")
print("-" * 50)
# Two independent coin flips
coin1 = [0.5, 0.5]  # Fair coin
coin2 = [0.6, 0.4]  # Biased coin

H_coin1 = entropy(coin1)
H_coin2 = entropy(coin2)

# Joint distribution for independent events: P(X,Y) = P(X) * P(Y)
joint = [coin1[i] * coin2[j] for i in range(2) for j in range(2)]
H_joint = entropy(joint)

print(f"  Coin 1:            H = {H_coin1:.4f} bits")
print(f"  Coin 2:            H = {H_coin2:.4f} bits")
print(f"  Sum:               H₁ + H₂ = {H_coin1 + H_coin2:.4f} bits")
print(f"  Joint (independent): H(X,Y) = {H_joint:.4f} bits")
print(f"  Difference: {abs(H_joint - (H_coin1 + H_coin2)):.6f} (should be ~0) ✓\n")

# Property 7: Conditioning Reduces Entropy
print("Property 7: Conditioning Reduces Entropy")
print("-" * 50)
print("  This is the key property for prediction!")
print("  Knowing context can only reduce uncertainty about what comes next.")
print("  We'll apply this in Session 3 with real sequence data:\n")
print("  - H(next move) without knowing previous = higher entropy")
print("  - H(next move | previous move) = lower entropy")
print("  - The reduction tells us how much context helps predict the next outcome\n")

print("Key Takeaway:")
print("   These properties explain why entropy is the 'right' way to measure uncertainty.")
print("   Any other measure that satisfies these axioms must be equivalent to Shannon entropy.")

---

## Part 1: Cross-Entropy — Coding with Wrong Assumptions

_Corresponds to lesson sections 4-6_

In Part 0, entropy H(P) told us the optimal code length when we know the true distribution P. But what if we design a code for distribution Q, then try to encode data from distribution P?

This is **cross-entropy** H(P,Q) — the average code length when using the wrong codebook.

### Exercise 9: The Coding Problem

_Lesson section 4: What is cross-entropy?_

**Scenario:** You're compressing English text, but you mistakenly use a codebook designed for French letter frequencies.

- True distribution P: English letter frequencies
- Your codebook Q: Designed for French frequencies  
- Result: You waste bits because common English letters get long codes

**Example with 3 letters:**
- True (English): P = [0.7, 0.2, 0.1]  (letter 'e' is very common)
- Optimal code for P: 'e'=0 (1 bit), others longer
- Wrong assumption Q = [0.33, 0.33, 0.34] (uniform)
- Wrong code: all letters get ~1.58 bits (log₂(3))
- **You waste bits!**

In [ ]:
import numpy as np
from alhikmah_llms import Session3

# Visualize: true distribution vs two coding assumptions
Session3.plot_cross_entropy_intuition(
    true_probs=[0.7, 0.2, 0.1],
    predicted_bad=[0.33, 0.33, 0.34],
    predicted_good=[0.65, 0.25, 0.1],
    labels=['e', 'a', 't']
)
plt.show()

print("Key Insight:")
print("   The codebook closer to the true distribution uses fewer bits on average!")
print("   H(P,Q) measures average bits per symbol when coding P with codebook for Q")

### Exercise 10: Where Do the Extra Bits Come From?

_Lesson section 5: Breakdown by symbol_

Cross-entropy is computed as:

```
H(P,Q) = -∑ P(x) log₂ Q(x)
```

In natural log (which we'll use from now on):
```
H(P,Q) = -∑ P(x) ln Q(x)  [nats]
```

Each symbol contributes `-P(x) ln Q(x)` to the average. This is:
- **Frequency** P(x) — how often this symbol appears in your data
- **Code length** -ln Q(x) — bits needed in your (wrong) codebook

**Question:** Which symbols waste the most bits?

In [ ]:
# Breakdown: see which symbols contribute most
Session3.plot_surprise_breakdown(
    true_probs=[0.7, 0.2, 0.05, 0.05],
    predicted=[0.5, 0.3, 0.1, 0.1],
    labels=['e', 'a', 't', 'o']
)
plt.show()

print("\nKey Observation:")
print("   Frequent symbols (high P) contribute most to cross-entropy!")
print("   Getting the code wrong for common symbols is expensive.")

### Exercise 11: Computing Cross-Entropy

_Lesson section 5: The formula_

Let's implement it and verify the intuition.

In [ ]:
def cross_entropy(P: list[float], Q: list[float]) -> float:
    """Compute H(P,Q) = -∑ P(x) ln Q(x)
    
    Args:
        P: True distribution (data frequencies)
        Q: Assumed distribution (codebook design)
    
    Returns:
        Average bits per symbol (in nats)
    """
    return -sum(p * np.log(q) if q > 0 else float('inf') for p, q in zip(P, Q))


# Test cases
P_true = [0.7, 0.2, 0.1]  # Real data

# Perfect codebook (Q = P)
Q_perfect = [0.7, 0.2, 0.1]
print(f"Perfect codebook (Q=P): H(P,Q) = {cross_entropy(P_true, Q_perfect):.4f} nats")

# Good codebook (Q ≈ P)
Q_good = [0.65, 0.25, 0.1]
print(f"Good codebook (Q≈P):    H(P,Q) = {cross_entropy(P_true, Q_good):.4f} nats")

# Bad codebook (Q uniform)
Q_bad = [0.33, 0.33, 0.34]
print(f"Bad codebook (uniform): H(P,Q) = {cross_entropy(P_true, Q_bad):.4f} nats")

# Terrible codebook (Q backwards!)
Q_terrible = [0.1, 0.2, 0.7]
print(f"Terrible (backwards):   H(P,Q) = {cross_entropy(P_true, Q_terrible):.4f} nats")

print("\nNotice: More bits needed as Q diverges from P!")

### Exercise 12: Bits vs Nats

_Lesson section 6: Why natural log?_

**In Part 0, we used log₂ (bits). From now on, we use ln (nats).**

Why?
- Machine learning libraries use natural log (PyTorch, TensorFlow)
- Derivatives are cleaner: d/dx(ln x) = 1/x
- Standard in information theory papers

**Conversion:** bits = nats / ln(2) ≈ nats × 1.443

A "nat" is the amount of information gained when choosing between e ≈ 2.718 equally likely outcomes (instead of 2 for a bit).

In [ ]:
# Compare bits vs nats
P = [0.7, 0.2, 0.1]
Q = [0.65, 0.25, 0.1]

# In nats (natural log)
ce_nats = cross_entropy(P, Q)

# In bits (log2)
ce_bits = -sum(p * math.log2(q) if q > 0 else 0 for p, q in zip(P, Q))

print(f"Cross-entropy in NATS: {ce_nats:.4f}")
print(f"Cross-entropy in BITS: {ce_bits:.4f}")
print(f"Ratio: {ce_bits / ce_nats:.4f} ≈ 1 / ln(2) ≈ 1.443")

print("\nFrom now on, we use NATS (natural log).")

---

## Part 2: KL Divergence — Quantifying Wasted Bits

_Corresponds to lesson sections 7-8_

The fundamental decomposition:

```
H(P,Q) = H(P) + KL(P||Q)
```

Where:
- **H(P)** = optimal code length (entropy of true distribution)
- **KL(P||Q)** = extra bits wasted by using wrong codebook
- **H(P,Q)** = actual bits used with wrong codebook

**KL divergence** (Kullback-Leibler divergence) measures the coding penalty for wrong assumptions.

### Exercise 13: Visualizing the Decomposition

_Lesson section 7: The breakdown_

H(P) is the best you could possibly do. KL(P||Q) is how many extra bits you waste.

In [ ]:
Session3.plot_entropy_decomposition()
plt.show()

print("\nCritical Insight:")
print("   H(P) = irreducible minimum (best possible code)")
print("   KL(P||Q) = wasted bits due to wrong assumptions")
print("   Goal: make Q match P to minimize waste")

### Exercise 14: Computing KL Divergence

_Lesson section 8: The formula_

KL divergence measures the difference between two distributions:

```
KL(P||Q) = ∑ P(x) ln(P(x) / Q(x))
         = ∑ P(x) ln P(x) - ∑ P(x) ln Q(x)
         = -H(P) + H(P,Q)
```

Properties:
- Always non-negative: KL(P||Q) ≥ 0
- Zero only when P = Q (no waste)
- **NOT symmetric:** KL(P||Q) ≠ KL(Q||P)

In [ ]:
def entropy_nats(P: list[float]) -> float:
    """Compute H(P) = -∑ P(x) ln P(x)"""
    return -sum(p * np.log(p) for p in P if p > 0)

def kl_divergence(P: list[float], Q: list[float]) -> float:
    """Compute KL(P||Q) = ∑ P(x) ln(P(x) / Q(x))"""
    return sum(p * np.log(p / q) if p > 0 and q > 0 else 0 
               for p, q in zip(P, Q))

# Verify the decomposition
P = [0.7, 0.2, 0.1]
Q = [0.5, 0.3, 0.2]

h_p = entropy_nats(P)
h_pq = cross_entropy(P, Q)
kl_pq = kl_divergence(P, Q)

print("Verify H(P,Q) = H(P) + KL(P||Q):")
print(f"  H(P)      = {h_p:.6f} nats  (optimal code length)")
print(f"  KL(P||Q)  = {kl_pq:.6f} nats  (wasted bits)")
print(f"  H(P) + KL = {h_p + kl_pq:.6f} nats")
print(f"  H(P,Q)    = {h_pq:.6f} nats  (actual bits used)")
print(f"\n  Match? {abs(h_pq - (h_p + kl_pq)) < 1e-10} ✓")

### Exercise 15: KL Divergence is NOT Symmetric

_Lesson section 8: Asymmetry_

KL(P||Q) ≠ KL(Q||P) because **encoding direction matters**:

- **KL(P||Q)**: Data from P, codebook for Q — "How bad is it to use Q's codebook on P's data?"
- **KL(Q||P)**: Data from Q, codebook for P — "How bad is it to use P's codebook on Q's data?"

These are different questions!

In [ ]:
Session3.plot_kl_asymmetry(
    P=[0.8, 0.15, 0.05],  # Peaked
    Q=[0.4, 0.3, 0.3],     # Flat
    labels=['A', 'B', 'C']
)
plt.show()

print("\nKey Point:")
print("   KL(P||Q) penalizes Q for missing high-probability events in P")
print("   KL(Q||P) penalizes Q for predicting unlikely events")
print("   Different costs → different values!")

In [ ]:
# Numerical example
P = [0.8, 0.15, 0.05]  # Peaked
Q = [0.4, 0.3, 0.3]     # Flat

kl_pq = kl_divergence(P, Q)
kl_qp = kl_divergence(Q, P)

print(f"KL(P||Q) = {kl_pq:.4f} nats  (encode P with Q's codebook)")
print(f"KL(Q||P) = {kl_qp:.4f} nats  (encode Q with P's codebook)")
print(f"\nDifference: {abs(kl_pq - kl_qp):.4f} nats")
print("They're NOT equal!")

---

## Part 3: Building Intuition

_Corresponds to lesson sections 9-10_

The best way to internalize these concepts is to experiment with different distributions.

### Exercise 16: The Loss Surface

_Lesson section 9: Visualizing cross-entropy_

For a 3-outcome distribution, we can visualize how H(P,Q) changes as Q varies.

**Key observation:** Minimum is at Q = P (perfect match, no wasted bits).

In [ ]:
Session3.plot_cross_entropy_surface(
    true_probs=[0.7, 0.2, 0.1]
)
plt.show()

print("\nObservations:")
print("   - Red star (Q = P) is the global minimum")
print("   - Bits wasted increases as Q moves away from P")
print("   - Unique minimum: only one best codebook")

### Exercise 17: Experiment with Distributions

_Lesson section 10: Build intuition_

Try different scenarios and observe how cross-entropy changes.

In [ ]:
# Example 1: Confident but wrong
P = [0.9, 0.08, 0.02]
Q_wrong = [0.1, 0.1, 0.8]  # Backwards!

print("Example 1: Wrong assumptions")
print(f"  True P = {P}")
print(f"  Codebook Q = {Q_wrong}")
print(f"  H(P,Q) = {cross_entropy(P, Q_wrong):.4f} nats")
print(f"  KL(P||Q) = {kl_divergence(P, Q_wrong):.4f} nats wasted")
print()

# Example 2: Uncertain (uniform)
Q_uniform = [0.33, 0.33, 0.34]
print("Example 2: Uniform assumptions (hedging)")
print(f"  Codebook Q = {Q_uniform}")
print(f"  H(P,Q) = {cross_entropy(P, Q_uniform):.4f} nats")
print(f"  KL(P||Q) = {kl_divergence(P, Q_uniform):.4f} nats wasted")
print(f"  (Less waste than being confidently wrong!)")
print()

# Your turn: create more examples!
# What happens when Q is very close to P?
# What happens when Q assigns zero probability to something that appears in P?

### Exercise 18: Visualize Your Own Examples

_Lesson section 10: Custom distributions_

In [ ]:
# Create your own example
my_true = [0.6, 0.3, 0.1]
my_codebook_bad = [0.2, 0.4, 0.4]
my_codebook_good = [0.58, 0.32, 0.1]

Session3.plot_cross_entropy_intuition(
    true_probs=my_true,
    predicted_bad=my_codebook_bad,
    predicted_good=my_codebook_good,
    labels=['A', 'B', 'C']
)
plt.show()

---

## Summary and Connection to Machine Learning

### What We Learned

**Information Theory Fundamentals:**
1. **Entropy H(P)** = optimal compression size for distribution P (in nats or bits)
2. **Cross-entropy H(P,Q)** = actual size when compressing P with codebook designed for Q
3. **KL divergence KL(P||Q)** = wasted bits = H(P,Q) - H(P)
4. **Decomposition**: H(P,Q) = H(P) + KL(P||Q)
5. KL divergence is **not symmetric** because encoding direction matters
6. We use **nats** (natural log) instead of bits in ML

### Connection to Machine Learning

**Why does this matter for language models?**

In Session 3, you'll see that:
- **Prediction = Compression**: A model that predicts well can compress well
- **Cross-entropy becomes loss**: H(P,Q) measures how well model Q predicts data from P
- **Training minimizes waste**: Gradient descent reduces KL(P||Q) by making Q match P

The language of compression (codebooks, bit lengths) is the *same* as the language of prediction (probabilities, losses). This isn't a coincidence — Shannon showed they're equivalent.

### What's Next (Session 3)

Apply these concepts to real prediction problems:
- **Rock-Paper-Scissors:** Build transition matrices and measure their entropy
- **Character bigrams:** Predict next letters in names
- **The V² wall:** Why simple counting doesn't scale to real vocabularies
- **Loss curves:** Watch cross-entropy decrease during training

---

## Exercises for Practice

1. **Prove the decomposition**: Show that H(P,Q) = H(P) + KL(P||Q) algebraically
2. **Worst case**: For P = [0.7, 0.2, 0.1], what Q maximizes H(P,Q)? What about KL(P||Q)?
3. **Asymmetry**: Create distributions where KL(P||Q) >> KL(Q||P) or vice versa
4. **Huffman coding**: Implement a simple Huffman code and verify it achieves H(P) bits per symbol
5. **Morse code**: Analyze Morse code as a compression scheme. Does it match English letter frequencies?